# Step 2: Train YOLO26 on Swimming Pools

Trains YOLO26 nano, small, medium, and large via transfer learning on the manually-cleaned Roboflow export, then reports the metrics and failure cases required by the brief.

Runtime: A100 GPU, ~30 min for all four variants.

## Setup

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q "ultralytics>=8.4.52" supervision pandas matplotlib pyyaml
!yolo settings sync=False
import ultralytics; ultralytics.checks()

## Load dataset from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, pathlib, yaml

ZIP_PATH = '/content/drive/MyDrive/IE/CV/roboflow/Pools.yolov8.zip'
DATA_DIR = pathlib.Path('/content/dataset_hbb')

DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

# Rewrite data.yaml with absolute split paths so Ultralytics resolves them regardless of cwd.
DATA_YAML = DATA_DIR / 'data.yaml'
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg['path'] = str(DATA_DIR)
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
cfg['test']  = 'test/images'
DATA_YAML.write_text(yaml.safe_dump(cfg))
print(yaml.safe_dump(cfg))

## Training configuration

| Parameter | Value |
|---|---|
| Pretrained weights | yolo26n / s / m / l (COCO) |
| Optimizer | AdamW (explicit, so lr0 is respected) |
| Initial LR | 0.005 |
| LR scheduler | Cosine |
| Epochs | 100 |
| Patience | 30 |
| Image size | 640 |
| Batch size | 16 |
| Mixed precision | AMP |
| Augmentations | HSV (0.015 / 0.7 / 0.4), translate 0.1, scale 0.5, flip lr/ud, rotate 180, mosaic 1.0, mixup 0.05, close_mosaic last 10 epochs |
| Hardware | NVIDIA A100 40 GB |

In [ ]:
CFG = dict(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=30,
    optimizer='AdamW',
    lr0=0.005,
    cos_lr=True,
    close_mosaic=10,
    seed=0,
    deterministic=True,
    cache='ram',
    plots=True,
    amp=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    degrees=180,
    mosaic=1.0, mixup=0.05,
    exist_ok=True,
)
VARIANTS = ['yolo26n', 'yolo26s', 'yolo26m', 'yolo26l']

## Train all four variants

In [ ]:
from ultralytics import YOLO
import time, gc, torch

results = {}

for v in VARIANTS:
    print(f'\n=== {v} ===')
    gc.collect(); torch.cuda.empty_cache()
    t0 = time.time()
    model = YOLO(f'{v}.pt')
    train = model.train(name=v, **CFG)
    val = model.val(data=str(DATA_YAML), split='val', verbose=False)
    results[v] = {
        'mAP50':    float(val.box.map50),
        'mAP50_95': float(val.box.map),
        'precision': float(val.box.mp),
        'recall':    float(val.box.mr),
        'params':    sum(p.numel() for p in model.model.parameters()),
        'train_time_s': time.time() - t0,
        'best_weights': str(train.save_dir / 'weights' / 'best.pt'),
    }
    del model

## Comparison table and plots

In [ ]:
import pandas as pd

df = pd.DataFrame(results).T
df['params_M'] = df['params'] / 1e6
df = df[['mAP50', 'mAP50_95', 'precision', 'recall', 'params_M', 'train_time_s']].round(4)
df.columns = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'Params (M)', 'Train time (s)']
df.to_csv('/content/yolo26_comparison.csv')
df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(df.index, df['mAP@50-95'], color='#4C86E8')
axes[0].set_ylabel('mAP@50-95'); axes[0].set_title('Validation mAP@50-95')
for i, v in enumerate(df['mAP@50-95']):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center')

axes[1].scatter(df['Params (M)'], df['mAP@50-95'], s=120, c='#4C86E8')
for name, row in df.iterrows():
    axes[1].annotate(name, (row['Params (M)'], row['mAP@50-95']), xytext=(5, 5), textcoords='offset points')
axes[1].set_xlabel('Parameters (M)'); axes[1].set_ylabel('mAP@50-95'); axes[1].set_title('Accuracy vs model size')
plt.tight_layout(); plt.show()

## Best variant diagnostic plots

In [ ]:
from IPython.display import Image as IPyImage, display
import pathlib

BEST = df['mAP@50-95'].idxmax()
print(f'Best variant: {BEST}')
RUN_DIR = pathlib.Path(f'/content/runs/detect/{BEST}')
for f in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'val_batch0_pred.jpg']:
    p = RUN_DIR / f
    if p.exists():
        print(f); display(IPyImage(filename=str(p), width=700))

## Test-set evaluation (standard + TTA)

In [ ]:
best_model = YOLO(results[BEST]['best_weights'])

t_std = best_model.val(data=str(DATA_YAML), split='test', verbose=False)
print(f'TEST (no TTA):  mAP50={t_std.box.map50:.4f}  mAP50-95={t_std.box.map:.4f}  P={t_std.box.mp:.4f}  R={t_std.box.mr:.4f}')

t_tta = best_model.val(data=str(DATA_YAML), split='test', augment=True, verbose=False)
print(f'TEST (TTA on):  mAP50={t_tta.box.map50:.4f}  mAP50-95={t_tta.box.map:.4f}  P={t_tta.box.mp:.4f}  R={t_tta.box.mr:.4f}')

## Failure analysis (FPs + FNs)

Runs the best model on the test set, matches predictions to ground truth by IoU, and dumps the highest-confidence false positives and the largest missed pools (false negatives). Inspect the gallery to write the discussion in the report.

In [ ]:
import cv2, numpy as np, pathlib, matplotlib.pyplot as plt

TEST_IMG_DIR = DATA_DIR / 'test' / 'images'
TEST_LBL_DIR = DATA_DIR / 'test' / 'labels'

def load_yolo_labels(p, w, h):
    """Roboflow's segmentation-type 'YOLOv8' export emits class + polygon
    vertices per line, so handle both 4-coord bbox and 2N-coord polygon lines.
    Returns (axis-aligned bbox, polygon points) per object."""
    out = []
    if not p.exists():
        return out
    for line in p.read_text().splitlines():
        toks = line.split()
        if len(toks) < 5: continue
        coords = list(map(float, toks[1:]))
        if len(coords) == 4:
            cx, cy, bw, bh = coords
            x1 = (cx - bw/2) * w; y1 = (cy - bh/2) * h
            x2 = (cx + bw/2) * w; y2 = (cy + bh/2) * h
            poly = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
        else:
            xs = np.array([coords[i] * w for i in range(0, len(coords), 2)])
            ys = np.array([coords[i] * h for i in range(1, len(coords), 2)])
            x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
            poly = np.stack([xs, ys], axis=1)
        out.append(([x1, y1, x2, y2], poly))
    return out

def iou(a, b):
    xA = max(a[0], b[0]); yA = max(a[1], b[1])
    xB = min(a[2], b[2]); yB = min(a[3], b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0

IOU_TH = 0.5
fps, fns = [], []

for img_p in sorted(TEST_IMG_DIR.iterdir()):
    if img_p.suffix.lower() not in {'.jpg', '.jpeg', '.png'}: continue
    img = cv2.imread(str(img_p)); h, w = img.shape[:2]
    gt = load_yolo_labels(TEST_LBL_DIR / (img_p.stem + '.txt'), w, h)
    gt_boxes = [g[0] for g in gt]; gt_polys = [g[1] for g in gt]

    r = best_model.predict(str(img_p), conf=0.25, verbose=False)[0]
    pred_boxes = r.boxes.xyxy.cpu().numpy()
    pred_conf  = r.boxes.conf.cpu().numpy()

    # Best-IoU greedy matching, iterating preds by descending confidence.
    matched_gt, matched_pr = set(), set()
    for i in np.argsort(-pred_conf):
        best_j, best_iou = -1, IOU_TH
        for j, gb in enumerate(gt_boxes):
            if j in matched_gt: continue
            v = iou(pred_boxes[i], gb)
            if v >= best_iou: best_iou, best_j = v, j
        if best_j >= 0:
            matched_gt.add(best_j); matched_pr.add(int(i))

    # FPs: also record best-IoU vs any GT, so we can split true alarms from loc errors.
    for i in range(len(pred_boxes)):
        if i in matched_pr: continue
        best = max((iou(pred_boxes[i], gb) for gb in gt_boxes), default=0)
        fps.append({'img': img_p, 'pred': pred_boxes[i].tolist(), 'conf': float(pred_conf[i]),
                    'best_iou_vs_gt': float(best), 'gt_polys': gt_polys})

    # FNs: best-IoU vs any prediction (matched or not), to flag loc errors vs true misses.
    for j, gb in enumerate(gt_boxes):
        if j in matched_gt: continue
        best = max((iou(gb, pred_boxes[i].tolist()) for i in range(len(pred_boxes))), default=0)
        fns.append({'img': img_p, 'gt_box': gb, 'gt_poly': gt_polys[j],
                    'area': (gb[2]-gb[0])*(gb[3]-gb[1]),
                    'best_iou_vs_pred': float(best),
                    'pred_boxes': pred_boxes.tolist()})

fps.sort(key=lambda x: -x['conf'])
fns.sort(key=lambda x: -x['area'])

fp_true   = sum(1 for f in fps if f['best_iou_vs_gt'] < 0.1)
fp_locerr = sum(1 for f in fps if f['best_iou_vs_gt'] >= 0.1)
fn_true   = sum(1 for f in fns if f['best_iou_vs_pred'] < 0.1)
fn_locerr = sum(1 for f in fns if f['best_iou_vs_pred'] >= 0.1)
print(f'False positives: {len(fps)}  (true alarms: {fp_true}, loc errors / near-miss: {fp_locerr})')
print(f'False negatives: {len(fns)}  (true misses: {fn_true}, loc errors / near-miss: {fn_locerr})')

In [ ]:
def draw_overlay(img, polys=None, boxes=None, focus_box=None, focus_poly=None):
    """GT polygons in green, all predictions in light blue, the FP/FN under
    inspection in red (thicker). Lets a reader tell loc errors from true misses."""
    out = cv2.cvtColor(cv2.imread(str(img)), cv2.COLOR_BGR2RGB)
    if polys is not None:
        for poly in polys:
            cv2.polylines(out, [poly.astype(np.int32)], isClosed=True, color=(0, 200, 0), thickness=2)
    if boxes is not None:
        for b in boxes:
            x1, y1, x2, y2 = map(int, b)
            cv2.rectangle(out, (x1, y1), (x2, y2), (80, 160, 255), 1)
    if focus_box is not None:
        x1, y1, x2, y2 = map(int, focus_box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (255, 0, 0), 3)
    if focus_poly is not None:
        cv2.polylines(out, [focus_poly.astype(np.int32)], isClosed=True, color=(255, 0, 0), thickness=3)
    return out

def show_fp_gallery(items, n=5):
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4.5))
    for ax, f in zip(axes, items[:n]):
        out = draw_overlay(f['img'], polys=f['gt_polys'], focus_box=f['pred'])
        tag = 'TRUE_FP' if f['best_iou_vs_gt'] < 0.1 else f"LOC_ERR(IoU={f['best_iou_vs_gt']:.2f})"
        ax.imshow(out)
        ax.set_title(f"{f['img'].name[:20]}\nconf={f['conf']:.2f}  {tag}", fontsize=8)
        ax.axis('off')
    fig.suptitle('Top-5 false positives  (green=GT polygon, red=FP prediction)')
    plt.tight_layout(); plt.show()

def show_fn_gallery(items, n=5):
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4.5))
    for ax, f in zip(axes, items[:n]):
        out = draw_overlay(f['img'], boxes=f['pred_boxes'], focus_poly=f['gt_poly'])
        tag = 'TRUE_MISS' if f['best_iou_vs_pred'] < 0.1 else f"LOC_ERR(IoU={f['best_iou_vs_pred']:.2f})"
        ax.imshow(out)
        ax.set_title(f"{f['img'].name[:20]}\narea={f['area']:.0f}  {tag}", fontsize=8)
        ax.axis('off')
    fig.suptitle('Top-5 false negatives  (red=missed GT polygon, blue=all predictions)')
    plt.tight_layout(); plt.show()

show_fp_gallery(fps)
show_fn_gallery(fns)

## Save results to Drive

In [ ]:
import shutil, pathlib

OUT = pathlib.Path('/content/drive/MyDrive/IE/CV/results/yolo26_hbb')
OUT.mkdir(parents=True, exist_ok=True)

shutil.copy('/content/yolo26_comparison.csv', OUT / 'comparison.csv')
for v in VARIANTS:
    src = pathlib.Path(results[v]['best_weights'])
    if src.exists():
        shutil.copy(src, OUT / f'{v}_best.pt')
print(f'Saved to {OUT}')